# 05 — Results tables and figures

This notebook **presents** the final results without changing the data-cleaning, coding, or thematic decisions. It reads the restricted cleaned survey and Notebook 04's final theme summary. Primary correlations and the centered moderation model are **recomputed** here to avoid stale hardcoded statistics; detailed inference and diagnostics belong in Notebook 03.

**Data units:** 124 complete quantitative records; 429 retained qualitative responses from 118 source respondents; 112 respondents represented in coded highlights (subject to confirmation against the latest Notebook 04 export). Subtheme evidence counts are **not prevalence estimates**. No quotation texts are loaded or exported here.

**Privacy:** Do not publish `data/processed/` or private Taguette exports.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf
from IPython.display import display

# Locate the project root regardless of the notebook folder.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (
            folder / "data" / "processed" / "survey_processed.csv"
        ).is_file()
        and (
            folder / "data" / "processed" / "qualitative"
            / "exports" / "06_final_theme_summary.csv"
        ).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate the required analysis inputs from {cwd}. "
        "Check that Notebooks 01 and 04 completed successfully."
    )

# Define input and output directories.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
QUAL_EXPORTS = PROCESSED_DIR / "qualitative" / "exports"

TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR = PROJECT_ROOT / "figures"

for folder in (TABLES_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Load quantitative and qualitative results.
survey_path = PROCESSED_DIR / "survey_processed.csv"
theme_path = QUAL_EXPORTS / "06_final_theme_summary.csv"

for path in (survey_path, theme_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required input: {path}. "
            "Run Notebooks 01 and 04 first."
        )

survey_df = pd.read_csv(survey_path)

final_theme_summary = (
    pd.read_csv(theme_path)
    .drop(columns=["Unnamed: 0"], errors="ignore")
)

# Validate quantitative analysis inputs.
required = [
    "gad7_total",
    "aise_mean",
    "llm_overall",
    "llm_task_mean",
]

assert all(c in survey_df.columns for c in required), (
    "Missing quantitative analysis variables"
)

assert len(survey_df) == 124, "Unexpected quantitative sample size"
assert survey_df[required].notna().all().all(), "Missing analysis scores"

assert survey_df["gad7_total"].between(0, 21).all()
assert survey_df["aise_mean"].between(1, 7).all()

assert survey_df[
    ["llm_overall", "llm_task_mean"]
].apply(lambda s: s.between(1, 5).all()).all()

# Validate the final qualitative theme structure.
needed_theme_cols = [
    "theme_id",
    "final_theme",
    "subtheme",
    "central_idea",
    "n_highlights",
    "n_participants",
]

assert set(needed_theme_cols).issubset(
    final_theme_summary.columns
), "Missing qualitative summary columns"

assert (
    len(final_theme_summary) == 8
    and final_theme_summary["theme_id"].nunique() == 4
), "Unexpected theme structure"

assert final_theme_summary["subtheme"].is_unique, (
    "Duplicate subtheme labels"
)

# Confirm successful loading and validation.
print("Project root:", PROJECT_ROOT)
print("Survey found:", survey_path.is_file())
print("Theme summary found:", theme_path.is_file())

print(
    "PASS: 124 quantitative records; "
    "4 final themes; 8 subthemes. "
    "Inputs present and valid."
)

## Descriptive and inferential results

Computed from the restricted cleaned dataset on each run. Thesis-reference assertions prevent the report from silently drifting away from the validated quantitative results.

In [ ]:
labels = {
    "gad7_total": "GAD-7 total",
    "aise_mean": "AI self-efficacy mean",
    "llm_overall": "Overall LLM use",
    "llm_task_mean": "Coding-task LLM use mean",
}
quantitative_descriptives = (
    survey_df[list(labels)]
    .describe().T
    .rename(columns={"count": "N", "mean": "Mean", "std": "SD", "50%": "Median", "25%": "Q1", "75%": "Q3", "min": "Min", "max": "Max"})
    [["N", "Mean", "SD", "Median", "Q1", "Q3", "Min", "Max"]]
    .rename(index=labels).rename_axis("Variable").reset_index()
)
reference = [(8.00, 5.85), (4.99, 0.89), (3.45, 1.22), (3.15, 1.08)]
for (_, row), (expected_mean, expected_sd) in zip(quantitative_descriptives.iterrows(), reference):
    assert abs(row["Mean"] - expected_mean) < .0051, f"Mean changed: {row['Variable']}"
    assert abs(row["SD"] - expected_sd) < .0051, f"SD changed: {row['Variable']}"
display(quantitative_descriptives.round(3))

In [ ]:
tests = [
    ("RQ1", "GAD-7 vs overall LLM use", "Spearman", "rho", spearmanr(survey_df.gad7_total, survey_df.llm_overall)),
    ("RQ1", "GAD-7 vs coding-task LLM use", "Pearson", "r", pearsonr(survey_df.gad7_total, survey_df.llm_task_mean)),
    ("RQ1", "GAD-7 vs coding-task LLM use", "Spearman", "rho", spearmanr(survey_df.gad7_total, survey_df.llm_task_mean)),
]
results = []
for rq, analysis, method, symbol, test in tests:
    results.append({"Research question": rq, "Analysis": analysis, "Method": method,
                    "Estimate": float(test.statistic), "Statistic": symbol,
                    "p": float(test.pvalue), "95% CI": "Not calculated here"})

model_data = survey_df[required].copy()
model_data["gad7_centered"] = model_data.gad7_total - model_data.gad7_total.mean()
model_data["aise_centered"] = model_data.aise_mean - model_data.aise_mean.mean()
model_data["gad7_x_aise"] = model_data.gad7_centered * model_data.aise_centered
model = smf.ols("llm_task_mean ~ gad7_centered + aise_centered + gad7_x_aise", data=model_data).fit()
interaction_b = float(model.params["gad7_x_aise"])
interaction_p = float(model.pvalues["gad7_x_aise"])
results.append({"Research question": "RQ2", "Analysis": "GAD-7 × AI self-efficacy",
                "Method": "Centered OLS moderation", "Estimate": interaction_b,
                "Statistic": "b", "p": interaction_p,
                "95% CI": "See Notebook 03 for bootstrap CI"})
quantitative_results_summary = pd.DataFrame(results)
expected = [(0.124, .170), (.140, .121), (.160, .076), (-.0102, .5682)]
for row, (effect, p) in zip(results, expected):
    assert abs(row["Estimate"] - effect) < .001, f"Estimate differs: {row['Analysis']}"
    assert abs(row["p"] - p) < .001, f"p-value differs: {row['Analysis']}"
quantitative_results_summary["Interpretation"] = np.where(
    quantitative_results_summary["p"] < .05, "Statistically significant", "Not statistically significant"
)
display(quantitative_results_summary.round(4))
print("PASS: RQ1 and RQ2 estimates/p-values agree with validated thesis reporting precision.")

## Final qualitative thematic structure

The categories and their wording are loaded from Notebook 04's final export, not recreated or renamed here. Counts indicate breadth of coded evidence, not population-level prevalence.

In [ ]:
qualitative_results = final_theme_summary[needed_theme_cols].copy()
qualitative_results_table = qualitative_results[["theme_id", "final_theme", "subtheme", "n_participants", "n_highlights"]].rename(
    columns={"theme_id": "Theme", "final_theme": "Final theme", "subtheme": "Subtheme",
             "n_participants": "Participants represented", "n_highlights": "Coded excerpts"}
)
display(qualitative_results_table)

## Figures

These are the four outputs from the original notebook: GAD-7 distribution, anxiety/LLM-use association, breadth across subthemes, and the final thematic map. The statistical annotation is generated from current data.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(survey_df.gad7_total, bins=range(23), edgecolor="black")
ax.axvline(survey_df.gad7_total.mean(), linestyle="--", linewidth=2,
           label=f"Mean = {survey_df.gad7_total.mean():.2f}")
ax.set(xlabel="GAD-7 total score", ylabel="Number of participants",
       title="Distribution of GAD-7 anxiety scores")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure_01_gad7_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
x, y = survey_df.gad7_total, survey_df.llm_task_mean
ax.scatter(x, y, alpha=.65)
slope, intercept = np.polyfit(x, y, 1)
xx = np.linspace(x.min(), x.max(), 100)
ax.plot(xx, slope * xx + intercept, linewidth=2)
ax.set(xlabel="GAD-7 total score", ylabel="Mean coding-task LLM use",
       title="Relationship between anxiety symptoms and coding-task LLM use")
ax.text(.03, .95,
        f"Pearson r = {results[1]['Estimate']:.3f}, p = {results[1]['p']:.3f}\n"
        f"Spearman ρ = {results[2]['Estimate']:.3f}, p = {results[2]['p']:.3f}",
        transform=ax.transAxes, va="top")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure_02_anxiety_llm_use_relationship.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sub = final_theme_summary.sort_values("n_participants")
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(sub.subtheme, sub.n_participants)
ax.set(xlabel="Participants represented", title="Breadth of coded evidence across subthemes")
ax.bar_label(ax.containers[0], padding=3)
ax.set_xlim(0, max(sub.n_participants) * 1.18)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure_03_subtheme_participant_breadth.png", dpi=300, bbox_inches="tight")
plt.show()

# The source export determines the theme and subtheme wording.
fig, ax = plt.subplots(figsize=(15, 8))
ax.set(xlim=(0, 15), ylim=(0, 10)); ax.axis("off")
def box(x, y, w, h, label, fontsize=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.025", fill=False, linewidth=1.2))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=fontsize, wrap=True)
box(5, 8.7, 5, .65, "Students' experiences of LLM use", 12)
for j, (_, group) in enumerate(final_theme_summary.groupby("theme_id", sort=True)):
    assert len(group) == 2, "Each final theme must have exactly 2 subthemes"
    x0 = .25 + j * 3.75
    short_title = group.iloc[0].final_theme
    # Line breaks are display-only; the source wording is not modified.
    import textwrap
    title = "\n".join(textwrap.wrap(short_title, 23))
    box(x0, 6.25, 3.45, 1.55, f"{group.iloc[0].theme_id}\n{title}", 8.5)
    ax.plot([7.5, x0+1.725], [8.7, 7.8], linewidth=.9)
    for k, (_, subrow) in enumerate(group.iterrows()):
        label = "\n".join(textwrap.wrap(subrow.subtheme, 27))
        yy = 3.6 - k * 2.05
        box(x0, yy, 3.45, 1.55, label, 8)
        ax.plot([x0+1.725, x0+1.725], [6.25, yy+1.55], linewidth=.9)
ax.set_title("Final thematic structure", fontsize=14, pad=20)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure_04_final_thematic_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
expected_counts = {
    "1a": 77,
    "1b": 53,
    "2a": 20,
    "2b": 20,
    "3a": 66,
    "3b": 54,
    "4a": 53,
    "4b": 27,
}

for prefix, expected in expected_counts.items():
    matches = final_theme_summary[
        final_theme_summary["subtheme"].str.startswith(prefix + ".")
    ]

    assert len(matches) == 1, f"Missing or duplicate subtheme: {prefix}"

    actual = int(matches.iloc[0]["n_participants"])
    assert actual == expected, (
        f"{prefix}: expected {expected}, got {actual}. "
        "Recheck Notebook 04 exports."
    )

print("PASS: all 8 subtheme counts match the corrected source audit.")

In [ ]:
quantitative_descriptives.to_csv(TABLES_DIR / "table_01_quantitative_descriptives.csv", index=False)
quantitative_results_summary.to_csv(TABLES_DIR / "table_02_quantitative_results.csv", index=False)
qualitative_results_table.to_csv(TABLES_DIR / "table_03_qualitative_themes.csv", index=False)
print("PASS: 3 aggregate tables and 4 figures exported.")
print("The full participant-level survey, Taguette DB, and quotation texts were not exported by this notebook.")
print("Still review all other repository files and Git history before any public push.")

In [ ]:
print("=== TABLE STRUCTURE VALIDATION ===")

for name, table in {
    "Descriptive statistics": quantitative_descriptives,
    "RQ1/RQ2 results": quantitative_results_summary,
    "Qualitative themes": qualitative_results_table,
}.items():
    print(f"\n{name}:")
    print("Columns:", table.columns.tolist())
    print("Shape:", table.shape)

print("\n=== QUALITATIVE EVIDENCE CHECK ===")
print("Final themes:", final_theme_summary["theme_id"].nunique())
print("Final subthemes:", final_theme_summary["subtheme"].nunique())

print("\nValidation completed.")
